<a href="https://colab.research.google.com/github/Krishishah7/nlp-learning-series/blob/main/06_llm_and_fine_tuning/11_topk_rag/topk_rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -U sentence-transformers transformers faiss-cpu sentencepiece --quiet

In [ ]:
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

embed_model = SentenceTransformer("all-MiniLM-L6-v2")

model_name = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

In [ ]:
documents = [
    "Paris is the capital city of France.",
    "France is located in Europe.",
    "The Eiffel Tower is a famous landmark in Paris.",
    "Berlin is the capital of Germany.",
    "Madrid is the capital of Spain."
]

In [ ]:
import faiss
import numpy as np

doc_embeddings = embed_model.encode(documents)

dimension = doc_embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)
index.add(np.array(doc_embeddings))

In [ ]:
question = "What is the capital of France and where is it located?"

question_embedding = embed_model.encode([question])

In [9]:
k = 3  # number of documents to retrieve

D, I = index.search(np.array(question_embedding), k)

retrieved_docs = [documents[i] for i in I[0]]

print("RETRIEVED DOCUMENTS:\n")
for doc in retrieved_docs:
    print("-", doc)

RETRIEVED DOCUMENTS:

- Paris is the capital city of France.
- France is located in Europe.
- The Eiffel Tower is a famous landmark in Paris.


In [10]:
context = " ".join(retrieved_docs)

In [11]:
prompt = f"""
Use the following context to answer the question.

Context: {context}

Question: {question}
"""

inputs = tokenizer(prompt, return_tensors="pt")

outputs = model.generate(**inputs, max_new_tokens=40)

answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("\nFINAL ANSWER:")
print(answer)


FINAL ANSWER:
Paris
